# Homegrown Audio Transformer

In [1]:
from my_engine.data import get_dataloaders
from src.my_engine.config import ModelConfig, TrainerConfig
from src.my_engine.utils import build_model, make_optimizer
from src.my_engine.audio import create_audio_datasets
from src.my_engine.trainer import Trainer
import torch
import os
import sys

In [2]:
PROJECT_PATH = '/home/alexsearle/Documents/Bucknell/SP26/Biometrics/biometrics_final_project'

print(f"Project path: \"{PROJECT_PATH}\"")
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

# Ensure that the current working directory is PROJECT_PATH
if os.getcwd() != PROJECT_PATH:
    print(f"Changing working directory to project path: \"{PROJECT_PATH}\"")
    os.chdir(PROJECT_PATH)
else:
    print(f"Already in the correct working directory: \"{PROJECT_PATH}\"")

Project path: "/home/alexsearle/Documents/Bucknell/SP26/Biometrics/biometrics_final_project"
Changing working directory to project path: "/home/alexsearle/Documents/Bucknell/SP26/Biometrics/biometrics_final_project"


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
train_ds, val_ds, test_ds, num_outputs = create_audio_datasets("data/wav", 16000, 100)

In [6]:
accel_device = torch.device("cuda:0")
speaker_transformer_config = ModelConfig(
    model_type="speaker_transformer",
    embedding_dim=256,
    dim_feedforward=1024,
    num_heads=8,
    num_encoder_layers=4,
    dropout=[0.3]
)

speaker_model = build_model(input_spec=1, config=speaker_transformer_config, num_outputs=num_outputs)

trainer_config = TrainerConfig(
    optimizer_name="adam",
    learning_rate=5e-4,
    weight_decay=1e-3,
    trainer_batch_size=8,
    evaluator_batch_size=64,
    device=accel_device,
    num_epochs=50,
    early_stopping_patience=None,
    use_scheduler=True,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds,
    val_ds,
    test_ds,
    train_batch_size=trainer_config.trainer_batch_size,
    eval_batch_size=trainer_config.evaluator_batch_size,
)

with Trainer(
    model=speaker_model,
    optimizer=make_optimizer(speaker_model.parameters(), trainer_config),
    criterion=torch.nn.CrossEntropyLoss(),
    config=trainer_config,
) as trainer:
    results = trainer.fit(train_loader, val_loader)

Epoch 0: Train Loss=3.5426,Val Loss=8.4624, Val Acc=3.42%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=2.7803,Val Loss=8.7251, Val Acc=3.34%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=2.3295,Val Loss=8.7565, Val Acc=3.26%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 3: Train Loss=1.9839,Val Loss=9.1152, Val Acc=3.42%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 4: Train Loss=1.6803,Val Loss=8.7862, Val Acc=3.42%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 5: Train Loss=1.2633,Val Loss=8.8467, Val Acc=3.42%,Metrics: Train: None, Test: None
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 6: Train Loss=1.1563,Val Loss=8.8189, Val Acc=3.42%,Metrics: Train:

In [10]:
best_model = torch.load("checkpoints/best.pt")
print(best_model.keys())
best_weights = best_model['model_state_dict']
speaker_model.load_state_dict(best_weights)

dict_keys(['model_state_dict', 'model_architecture', 'trainer_config', 'optimizer_state_dict', 'best_val_loss', 'epoch', 'patience_counter', 'scheduler_state_dict'])


<All keys matched successfully>

In [11]:
torch.save(speaker_model, "models/speaker_model.pt")